# TP3 — 02. Relaciones bivariadas del error meteorológico

**Taller de Análisis de Datos I · Versión revisada para el dashboard · 17/11/2026**

**Pregunta central:** ¿cómo se asocia el error del pronóstico de temperatura con las condiciones meteorológicas, la ubicación y la hora objetivo, diferenciando los horizontes de 12 y 24 horas?

**Definición:** `error_c = temperatura_observada − temperatura_pronosticada`. Un error **positivo** indica subestimación de temperatura; uno **negativo**, sobreestimación. `MAE = media(|error_c|)`.

**Población de desarrollo:** únicamente 48 días históricos, **10/09–01/11/2026**, excluyendo el 22–26/10 conforme a la recomendación docente. **02–09/11 es prueba reservada:** no se abre ni analiza aquí. La fecha **01/11 sí pertenece al histórico**; no debe confundirse con el comienzo del test.

**Límites:** análisis descriptivo y bivariado, no causal. Una asociación con el error no demuestra importancia predictiva ni que pueda usarse sin fugas. Tampoco todos los registros son independientes (estación, día y horizonte).

**Objetivo operativo:** tablas y gráficos reproducibles para construir la exposición **solo en el dashboard**; ningún gráfico del notebook será necesario durante la defensa.

## 1. Entorno, particiones y procedencia

Preferimos consumir el **Parquet histórico consolidado y auditado en el Notebook 00** para evitar leer nuevamente decenas de CSV. Si aún no se exportó, el notebook puede reconstruir la misma cohorte **solo** leyendo los 48 CSV históricos por bloques. En ambos casos se verifica el manifiesto `tp3_config.py`. No se carga el test reservado.

In [ ]:
# 02.1 — Bibliotecas y partición temporal única (no incluye noviembre/test)
from pathlib import Path
from datetime import datetime, timezone
import sys
import json
import hashlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from IPython.display import display

RAIZ_PROYECTO = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / 'dataset').is_dir() and (p / 'tp3_config.py').is_file()),
    None
)
if RAIZ_PROYECTO is None:
    raise FileNotFoundError(
        'No se encontró TP3/dataset/ junto a TP3/tp3_config.py. '
        'Abrí el notebook desde TP3/notebooks/ o TP3/.'
    )
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

from tp3_config import obtener_particiones, HISTORICO, TEST, EXCLUIDOS

particiones_02 = obtener_particiones(RAIZ_PROYECTO, exigir_61=True)
archivos_historicos_02 = particiones_02['historico']
assert tuple(len(particiones_02[k]) for k in
             ('inventario', 'historico', 'excluidos', 'test')) == (61, 48, 5, 8)
assert set(archivos_historicos_02).isdisjoint(particiones_02['test'])
assert set(archivos_historicos_02).isdisjoint(particiones_02['excluidos'])

HORIZONTES_02 = (12, 24)
CHUNKSIZE_02 = 100_000
UMBRAL_GRAFICO_02 = 100  # Visibilidad del gráfico; los grupos pequeños siguen exportados
RESULTADOS_02 = RAIZ_PROYECTO / 'resultados' / 'dashboard' / '02_relaciones_bivariadas'
PARQUET_00 = (RAIZ_PROYECTO / 'datos_modelado' / 'consolidados'
              / 'tp3_historico_interval_12_24.parquet')
MANIFIESTO_00 = (RAIZ_PROYECTO / 'resultados' / 'dashboard'
                / '00_calidad_datos' / '13_auditoria_kpis_resumen.json')
RUN_ID_02 = datetime.now(timezone.utc).strftime('tp3_02_%Y%m%dT%H%M%SZ')

print('Proyecto:', RAIZ_PROYECTO)
print('Inventario: 61 | Histórico: 48 | Excluidos: 5 | Test NO abierto: 8')
print('Histórico:', archivos_historicos_02[0].name, '→', archivos_historicos_02[-1].name)
print('Fuente preferida:', PARQUET_00 if PARQUET_00.exists() else '48 CSV (alternativa)')

## 2. Carga histórica, integridad y variable objetivo

Se utilizan exclusivamente los emparejamientos `match_type == "interval"`, horizontes 12/24 h y ambas temperaturas finitas. Los predictores meteorológicos pueden faltar: cada asociación se calcula con los registros físicamente válidos para **esa variable**, sin descartar globalmente filas por ausencias en otra variable.

Los errores de predicción y sus transformaciones se calculan **una única vez**. Se verifica la cohorte anterior (**854.009 registros**) y se comprueba que no haya fechas de test. La comparación temporal es por fecha objetivo, aunque la partición inicial provenga de la fecha del archivo.

In [ ]:
# 02.2 — Lectura preferente del Parquet 00, o lectura segura por bloques
COLUMNAS_02 = [
    'station_id', 'state', 'latitude', 'longitude',
    'snapshot_time_utc', 'target_time_utc', 'horizon_target_hours',
    'match_type', 'temperature_forecast_c', 'temperature_observed_c',
    'relative_humidity_forecast_pct', 'dewpoint_forecast_c',
    'wind_speed_forecast_kmh', 'precip_probability_forecast_pct'
]
NUMERICAS_02 = [
    'latitude', 'longitude', 'horizon_target_hours',
    'temperature_forecast_c', 'temperature_observed_c',
    'relative_humidity_forecast_pct', 'dewpoint_forecast_c',
    'wind_speed_forecast_kmh', 'precip_probability_forecast_pct'
]
CLAVES_02 = ['station_id', 'snapshot_time_utc',
             'target_time_utc', 'horizon_target_hours']

if PARQUET_00.is_file():
    df_bivariado = pd.read_parquet(PARQUET_00, columns=COLUMNAS_02)
    fuente_02 = 'Parquet histórico del Notebook 00'
else:
    fragmentos_02 = []
    for i, archivo in enumerate(archivos_historicos_02, 1):
        for bloque in pd.read_csv(
            archivo, usecols=COLUMNAS_02, chunksize=CHUNKSIZE_02,
            low_memory=False
        ):
            for c in NUMERICAS_02:
                bloque[c] = pd.to_numeric(bloque[c], errors='coerce')
            sel = (bloque['horizon_target_hours'].isin(HORIZONTES_02)
                   & bloque['match_type'].eq('interval')
                   & np.isfinite(bloque['temperature_forecast_c'])
                   & np.isfinite(bloque['temperature_observed_c']))
            if sel.any():
                fragmentos_02.append(bloque.loc[sel].copy())
        if i % 8 == 0:
            print('Archivos históricos leídos:', i, '/ 48')
    if not fragmentos_02:
        raise ValueError('No hubo registros históricos válidos.')
    df_bivariado = pd.concat(fragmentos_02, ignore_index=True)
    del fragmentos_02
    fuente_02 = 'Reconstrucción desde los 48 CSV históricos'

for c in NUMERICAS_02:
    df_bivariado[c] = pd.to_numeric(df_bivariado[c], errors='coerce')
for c in ['snapshot_time_utc', 'target_time_utc']:
    df_bivariado[c] = pd.to_datetime(df_bivariado[c], utc=True, errors='coerce')

assert df_bivariado['horizon_target_hours'].isin(HORIZONTES_02).all()
assert df_bivariado['match_type'].eq('interval').all()
assert np.isfinite(df_bivariado[
    ['temperature_forecast_c', 'temperature_observed_c']
].to_numpy(dtype=float)).all()
assert df_bivariado[CLAVES_02].notna().all().all()
assert not df_bivariado.duplicated(CLAVES_02).any(), 'Claves duplicadas: revisar 00.'
assert set(df_bivariado['target_time_utc'].dt.date).issubset(HISTORICO), (
    'El histórico contiene fechas objetivo fuera de la partición.'
)
assert not (set(df_bivariado['target_time_utc'].dt.date) & TEST)
assert len(df_bivariado) == 854009, (
    f'Cohorte distinta a la declarada: {len(df_bivariado)}. '
    'Reconciliar con Notebook 00 antes de analizar.'
)
assert (df_bivariado['horizon_target_hours'].value_counts().to_dict()
        == {12:427009, 24:427000})

df_bivariado['error_c'] = (df_bivariado['temperature_observed_c']
                            - df_bivariado['temperature_forecast_c'])
df_bivariado['error_abs_c'] = df_bivariado['error_c'].abs()
df_bivariado['error_sq_c2'] = df_bivariado['error_c'].pow(2)
df_bivariado['fecha_objetivo'] = df_bivariado['target_time_utc'].dt.date
df_bivariado['hora_utc'] = df_bivariado['target_time_utc'].dt.hour

# Si el Parquet proviene del 00 actual, comprobar también las claves.
if MANIFIESTO_00.is_file():
    with open(MANIFIESTO_00, encoding='utf-8') as f:
        meta_00 = json.load(f)
    assert int(meta_00['filas_historicas_interval']) == len(df_bivariado)
    if PARQUET_00.is_file() and 'historico_claves_sha256' in meta_00:
        claves_ordenadas = df_bivariado.sort_values(
            ['target_time_utc', 'station_id', 'horizon_target_hours']
        ).reset_index(drop=True)
        hashes = pd.util.hash_pandas_object(
            claves_ordenadas[CLAVES_02], index=False
        ).to_numpy(dtype='uint64')
        huella_02 = hashlib.sha256(hashes.tobytes()).hexdigest()
        assert huella_02 == meta_00['historico_claves_sha256'], (
            'El Parquet histórico no coincide con el manifiesto actual del Notebook 00.'
        )
        print('Huella de cohorte reconciliada con Notebook 00.')
else:
    print('Manifiesto del 00 no disponible: se verifican claves/fechas y recuentos.')

print('Fuente:', fuente_02)
print('Registros:', len(df_bivariado))
print('Estaciones:', df_bivariado['station_id'].nunique())
print('Memoria estimada: %.1f MiB' % (
    df_bivariado.memory_usage(deep=True).sum() / 1024**2
))
display(df_bivariado.groupby('horizon_target_hours').agg(
    registros=('error_c', 'size'),
    sesgo_medio_c=('error_c', 'mean'),
    mae_c=('error_abs_c', 'mean')
).round(4))

## 3. Disponibilidad y validez física de variables

La auditoría registra **faltantes, no finitos y valores fuera de dominio**. Los rangos impuestos para correlaciones son: humedad y probabilidad de precipitación `[0, 100] %`; viento `≥0 km/h`. No se elimina una fila completa por un problema de otra variable. Valores altos pero plausibles en eventos extremos no se descartan arbitrariamente. Punto de rocío superior a la temperatura pronosticada se **señala**, no se elimina: puede reflejar inconsistencias entre productos o pequeñas inversiones ya contempladas en el modelado.

**Advertencia:** presión *observada* no se utiliza como predictor aquí ni debe introducirse sin verificar su disponibilidad en el instante del pronóstico.

In [ ]:
# 02.3 — Catálogo común y diagnóstico global de calidad
VARIABLES_02 = {
    'Temperatura pronosticada': {
        'columna':'temperature_forecast_c', 'unidad':'°C',
        'bordes':[-np.inf,0,10,20,30,np.inf],
        'etiquetas':['<0','0–10','10–20','20–30','≥30'], 'derecha':False
    },
    'Humedad relativa': {
        'columna':'relative_humidity_forecast_pct', 'unidad':'%',
        'bordes':[0,20,40,60,80,100],
        'etiquetas':['0–20','20–40','40–60','60–80','80–100'], 'derecha':True,
        'minimo':0, 'maximo':100
    },
    'Punto de rocío': {
        'columna':'dewpoint_forecast_c', 'unidad':'°C',
        'bordes':[-np.inf,-15,-10,-5,0,5,10,15,20,25,30,np.inf],
        'etiquetas':['<−15','−15 a −10','−10 a −5','−5 a 0',
                     '0–5','5–10','10–15','15–20','20–25','25–30','≥30'],
        'derecha':False
    },
    'Velocidad del viento': {
        'columna':'wind_speed_forecast_kmh', 'unidad':'km/h',
        'bordes':[0,5,10,15,20,30,40,np.inf],
        'etiquetas':['0–5','5–10','10–15','15–20','20–30','30–40','≥40'],
        'derecha':False, 'minimo':0
    },
    'Probabilidad de precipitación': {
        'columna':'precip_probability_forecast_pct', 'unidad':'%',
        'bordes':[-.000001,0,10,25,50,75,100],
        'etiquetas':['0','>0–10','>10–25','>25–50','>50–75','>75–100'],
        'derecha':True, 'minimo':0, 'maximo':100
    }
}

def mascara_fisica_02(tabla, especificacion):
    s = tabla[especificacion['columna']]
    ok = np.isfinite(s.to_numpy(dtype=float))
    if 'minimo' in especificacion:
        ok &= (s >= especificacion['minimo']).to_numpy()
    if 'maximo' in especificacion:
        ok &= (s <= especificacion['maximo']).to_numpy()
    return pd.Series(ok, index=tabla.index)

filas_calidad_02 = []
for nombre, spec in VARIABLES_02.items():
    c = spec['columna']
    for h in HORIZONTES_02:
        t = df_bivariado.loc[df_bivariado['horizon_target_hours'].eq(h)]
        numerico = t[c].to_numpy(dtype=float)
        es_nulo = t[c].isna().to_numpy()
        es_finito = np.isfinite(numerico)
        validos = mascara_fisica_02(t, spec).to_numpy()
        filas_calidad_02.append({
            'horizonte_h':h, 'variable':nombre, 'columna':c,
            'registros_total':len(t), 'faltantes':int(es_nulo.sum()),
            'no_finitos_no_nulos':int((~es_finito & ~es_nulo).sum()),
            'fuera_rango_fisico':int((es_finito & ~validos).sum()),
            'registros_validos':int(validos.sum()),
            'disponibilidad_valida_pct':round(100 * validos.mean(), 4)
        })

df_calidad_02 = pd.DataFrame(filas_calidad_02)
print('Disponibilidad por variable y horizonte:')
display(df_calidad_02.round(3))

dew = df_bivariado['dewpoint_forecast_c'].to_numpy(dtype=float)
temp = df_bivariado['temperature_forecast_c'].to_numpy(dtype=float)
inversion = np.isfinite(dew) & np.isfinite(temp) & (dew > temp)
print('Punto de rocío mayor que temperatura pronosticada (alerta, no exclusión):',
      int(inversion.sum()))
print('Vientos > 400 km/h (alerta de plausibilidad, no filtro automático):',
      int((df_bivariado['wind_speed_forecast_kmh'] > 400).sum()))

## 4. Análisis bivariado común: correlaciones, sesgo y MAE

Para cada una de las cinco variables se calculan Pearson y Spearman frente al **error firmado** y el **error absoluto**, separadamente para 12 y 24 horas. Las correlaciones usan una muestra de pares válidos para cada predictor; no deben compararse como si midieran causalidad o independencia entre filas. Las tablas por intervalos contienen **número de registros, estaciones y días** para revelar problemas de soporte.

La función compartida evita repetir lógica o aplicar filtros distintos en precipitación, humedad y en la síntesis global. Los rangos son descriptivos y no son umbrales aprendidos del test.

In [ ]:
# 02.4 — Función única para los cinco análisis y síntesis consistente

def analizar_variable_bivariada_02(tabla, nombre, spec):
    col = spec['columna']
    seleccion = mascara_fisica_02(tabla, spec)
    datos = tabla.loc[seleccion, [
        col, 'horizon_target_hours', 'station_id', 'fecha_objetivo',
        'error_c', 'error_abs_c', 'error_sq_c2'
    ]].copy()
    datos['rango'] = pd.cut(
        datos[col], bins=spec['bordes'], labels=spec['etiquetas'],
        right=spec['derecha'], include_lowest=True
    )
    if datos['rango'].isna().any():
        raise AssertionError(f'Un valor aceptado de {nombre} quedó fuera de sus bins.')

    corr = []
    for h in HORIZONTES_02:
        t = datos.loc[datos['horizon_target_hours'].eq(h)]
        x = t[col]
        corr.append({
            'variable':nombre, 'columna':col, 'horizonte_h':h,
            'registros':len(t), 'estaciones':t['station_id'].nunique(),
            'dias':t['fecha_objetivo'].nunique(),
            'pearson_error':x.corr(t['error_c'], method='pearson'),
            'spearman_error':x.corr(t['error_c'], method='spearman'),
            'pearson_error_abs':x.corr(t['error_abs_c'], method='pearson'),
            'spearman_error_abs':x.corr(t['error_abs_c'], method='spearman')
        })

    resumen = (datos.groupby(['horizon_target_hours','rango'], observed=True)
        .agg(registros=('error_c','size'), estaciones=('station_id','nunique'),
             dias=('fecha_objetivo','nunique'), sesgo_medio_c=('error_c','mean'),
             mae_c=('error_abs_c','mean'), mse_c2=('error_sq_c2','mean'))
        .reset_index())
    resumen['rmse_c'] = np.sqrt(resumen.pop('mse_c2'))
    resumen['variable'] = nombre
    resumen['unidad_predictor'] = spec['unidad']
    resumen['orden_rango'] = resumen['rango'].cat.codes
    resumen['rango'] = resumen['rango'].astype(str)
    resumen['grupo_soporte_grafico'] = resumen['registros'].ge(UMBRAL_GRAFICO_02)
    resumen['pct_en_horizonte_variable'] = (resumen['registros']
        / resumen.groupby('horizon_target_hours')['registros'].transform('sum') * 100)
    assert resumen['registros'].sum() == len(datos), f'{nombre}: grupos incompletos.'
    return pd.DataFrame(corr), resumen

correlaciones_02 = []
rangos_02 = []
for nombre_02, spec_02 in VARIABLES_02.items():
    c, r = analizar_variable_bivariada_02(df_bivariado, nombre_02, spec_02)
    correlaciones_02.append(c)
    rangos_02.append(r)

df_correlaciones_02 = pd.concat(correlaciones_02, ignore_index=True)
df_rangos_02 = pd.concat(rangos_02, ignore_index=True)
df_correlaciones_02['abs_pearson_error'] = df_correlaciones_02['pearson_error'].abs()

assert len(df_correlaciones_02) == len(VARIABLES_02) * 2
assert (df_rangos_02.groupby(['variable','horizon_target_hours'])['registros']
        .sum().sort_index().to_numpy()
        == df_correlaciones_02.set_index(['variable','horizonte_h'])
        .sort_index()['registros'].to_numpy()).all()

print('Correlaciones (todas con filtros físicos coherentes):')
display(df_correlaciones_02.round(4))
print('\nGrupos descriptivos de los cinco predictores:')
display(df_rangos_02.head(20).round(3))
print('Nota: bajo número de días/estaciones implica precaución, aunque haya muchas filas.')

## 5. Visualización reproducible: cinco variables meteorológicas

Los intervalos de viento y precipitación no tienen el mismo ancho; por eso utilizamos **barras categóricas**, no líneas cuya pendiente aparentaría una separación uniforme. Los gráficos muestran las dos preguntas distintas: **sesgo** (signo y calibración) y **MAE** (magnitud del error).

Se muestran solo grupos de al menos 100 registros por horizonte. Todos, incluso los grupos pequeños, permanecen en las tablas exportadas para que el dashboard pueda informar su soporte sin inventar observaciones. El umbral solo regula la visualización, no la selección del modelado.

In [ ]:
# 02.5 — Gráficos de sesgo y MAE con barras y ejes interpretables

def grafico_rangos_02(resumen, nombre, spec):
    t = resumen.loc[resumen['variable'].eq(nombre) &
                    resumen['grupo_soporte_grafico']].copy()
    categorias = list(spec['etiquetas'])
    x = np.arange(len(categorias), dtype=float)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, metrica, titulo in [
        (axes[0], 'sesgo_medio_c', 'Sesgo medio (observado − pronosticado)'),
        (axes[1], 'mae_c', 'Error absoluto medio (MAE)')
    ]:
        for desplazamiento, h in [(-0.19, 12), (0.19, 24)]:
            th = (t.loc[t['horizon_target_hours'].eq(h)]
                  .set_index('rango').reindex(categorias))
            ax.bar(x + desplazamiento, th[metrica], 0.37,
                   label=f'{h} h')
        if metrica == 'sesgo_medio_c':
            ax.axhline(0, color='black', linewidth=1, linestyle='--')
        ax.set_title(titulo)
        ax.set_xlabel(f'{nombre} ({spec["unidad"]})')
        ax.set_ylabel('Temperatura (°C)')
        ax.set_xticks(x, categorias, rotation=35, ha='right')
        ax.grid(axis='y', alpha=.25)
        ax.set_axisbelow(True)
        ax.legend(title='Horizonte')
    fig.suptitle(f'Error de temperatura según {nombre.lower()} — histórico 12/24 h')
    fig.tight_layout()
    return fig

RESULTADOS_02.mkdir(parents=True, exist_ok=True)
for i, (nombre, spec) in enumerate(VARIABLES_02.items(), start=1):
    fig = grafico_rangos_02(df_rangos_02, nombre, spec)
    fig.savefig(RESULTADOS_02 / f'10_{i:02d}_sesgo_mae_{spec["columna"]}.png',
                dpi=170, bbox_inches='tight')
    plt.show()
    plt.close(fig)
print('Cinco gráficos meteorológicos guardados. Cada barra conserva su n en CSV.')

## 6. Dependencias entre predictores y posibles factores de confusión

La temperatura, el rocío, la humedad y la **depresión del punto de rocío** pueden contener información redundante. Esto es especialmente relevante para interpretar coeficientes de Ridge y permutaciones de HGB en el Notebook 08. Una asociación bivariada puede variar dentro de estratos de hora o horizonte.

Se incluye una matriz descriptiva de predictores y un contraste **humedad × bloque horario UTC**. No son pruebas causales, ni equivalen a interpretabilidad del modelo entrenado. Las escalas y métodos se indican explícitamente.

In [ ]:
# 02.6 — Dependencia entre predictores: Pearson por horizonte y derivadas
columnas_pred_02 = [spec['columna'] for spec in VARIABLES_02.values()]
df_pred_limpias_02 = df_bivariado[
    ['horizon_target_hours'] + columnas_pred_02
].copy()
for nombre, spec in VARIABLES_02.items():
    c = spec['columna']
    df_pred_limpias_02.loc[~mascara_fisica_02(df_pred_limpias_02, spec), c] = np.nan

df_pred_limpias_02['depresion_rocio_c'] = (
    df_pred_limpias_02['temperature_forecast_c']
    - df_pred_limpias_02['dewpoint_forecast_c']
)
vars_corr_02 = columnas_pred_02 + ['depresion_rocio_c']
mats_02 = []
for h in HORIZONTES_02:
    matriz = df_pred_limpias_02.loc[
        df_pred_limpias_02['horizon_target_hours'].eq(h), vars_corr_02
    ].corr(method='pearson')
    for a in vars_corr_02:
        for b in vars_corr_02:
            mats_02.append({
                'horizonte_h':h, 'variable_a':a, 'variable_b':b,
                'pearson':matriz.loc[a, b]
            })
    print(f'Correlación entre predictores — {h} h:')
    display(matriz.round(3))

df_dependencias_02 = pd.DataFrame(mats_02)

# Matriz estática para consulta; el dashboard usará la tabla larga.
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, h in zip(axes, HORIZONTES_02):
    t = df_pred_limpias_02.loc[
        df_pred_limpias_02['horizon_target_hours'].eq(h), vars_corr_02
    ].corr()
    im = ax.imshow(t.to_numpy(), cmap='coolwarm', vmin=-1, vmax=1)
    etiquetas = ['Temperatura','Humedad','Rocío','Viento','Precip.','Depresión']
    ax.set_xticks(range(len(etiquetas)), etiquetas, rotation=55, ha='right')
    ax.set_yticks(range(len(etiquetas)), etiquetas)
    ax.set_title(f'Pearson — {h} h')
fig.colorbar(im, ax=axes, shrink=.65, label='Coeficiente de Pearson')
fig.suptitle('Dependencias entre predictores (pares disponibles; sin causalidad)', y=1.03)
fig.subplots_adjust(left=.16, right=.88, bottom=.20, wspace=.45)
fig.savefig(RESULTADOS_02 / '11_dependencias_predictores.png',
            dpi=170, bbox_inches='tight')
plt.show()
plt.close(fig)

In [ ]:
# 02.7 — Contraste exploratorio condicionado por hora UTC y horizonte
s_hum = VARIABLES_02['Humedad relativa']
cond = df_bivariado.loc[mascara_fisica_02(df_bivariado, s_hum), [
    'horizon_target_hours', 'hora_utc', 'station_id', 'fecha_objetivo',
    s_hum['columna'], 'error_c', 'error_abs_c'
]].copy()
cond['bloque_utc'] = pd.cut(
    cond['hora_utc'], bins=[-1,5,11,17,23],
    labels=['00–05','06–11','12–17','18–23']
)
cond['rango_humedad'] = pd.cut(
    cond[s_hum['columna']], bins=s_hum['bordes'],
    labels=s_hum['etiquetas'], include_lowest=True
)
df_estratos_02 = (cond.groupby(
    ['horizon_target_hours','bloque_utc','rango_humedad'], observed=True
).agg(registros=('error_c','size'), estaciones=('station_id','nunique'),
      dias=('fecha_objetivo','nunique'), sesgo_medio_c=('error_c','mean'),
      mae_c=('error_abs_c','mean')).reset_index())
for col in ['bloque_utc', 'rango_humedad']:
    df_estratos_02[col] = df_estratos_02[col].astype(str)
print('Humedad por bloque UTC (grupos con soporte variable):')
display(df_estratos_02.round(3))
print('Las diferencias entre bloques NO prueban efecto causal de la humedad.')

## 7. Dimensión temporal y geográfica exploratoria

El **Notebook 01** ya desarrolla cobertura y mapas de estaciones. Este apartado agrega un enfoque diferente: asociación del error con **hora UTC**, **estado**, **latitud** y **longitud**. Las métricas geográficas representan mezclas distintas de estaciones, fechas y condiciones meteorológicas; no deben interpretarse como un efecto causal de la coordenada. La hora UTC no es hora solar: la comparación física más completa se mantiene en el Notebook 03.

Los rankings estatales muestran su soporte (cantidad de estaciones y días) y no se extrapolan automáticamente a estaciones desconocidas.

In [ ]:
# 02.8 — Hora UTC, regiones y bandas geográficas

def agregar_error_02(tabla, columnas):
    tmp = (tabla.groupby(columnas, observed=True, dropna=False)
        .agg(registros=('error_c','size'), estaciones=('station_id','nunique'),
             dias=('fecha_objetivo','nunique'), sesgo_medio_c=('error_c','mean'),
             mae_c=('error_abs_c','mean'), mse_c2=('error_sq_c2','mean'))
        .reset_index())
    tmp['rmse_c'] = np.sqrt(tmp.pop('mse_c2'))
    return tmp

df_horario_02 = agregar_error_02(
    df_bivariado, ['horizon_target_hours','hora_utc'])

estados_02 = df_bivariado.copy()
estados_02['state'] = estados_02['state'].fillna('Sin estado').replace('', 'Sin estado')
df_estados_02 = agregar_error_02(
    estados_02, ['horizon_target_hours','state'])

geo_02 = df_bivariado.loc[
    df_bivariado['latitude'].between(-90,90)
    & df_bivariado['longitude'].between(-180,180)
].copy()
geo_02['banda_latitud'] = pd.cut(
    geo_02['latitude'], bins=[-90,25,30,35,40,45,50,90],
    include_lowest=True)
geo_02['banda_longitud'] = pd.cut(
    geo_02['longitude'], bins=[-180,-125,-115,-105,-95,-85,-75,-65,180],
    include_lowest=True)
df_latitud_02 = agregar_error_02(geo_02, ['horizon_target_hours','banda_latitud'])
df_longitud_02 = agregar_error_02(geo_02, ['horizon_target_hours','banda_longitud'])
for t, nombre in [(df_latitud_02,'banda_latitud'), (df_longitud_02,'banda_longitud')]:
    t[nombre] = t[nombre].astype(str)

# Asociación geográfica sobre estación agregada, para no inflar independencia.
df_estacion_geo_02 = agregar_error_02(
    geo_02, ['horizon_target_hours','station_id'])
coords_02 = (geo_02.groupby('station_id', as_index=False)
             .agg(latitude=('latitude','median'),longitude=('longitude','median')))
df_estacion_geo_02 = df_estacion_geo_02.merge(
    coords_02, on='station_id', how='left', validate='many_to_one')
elegibles_geo_02 = df_estacion_geo_02.loc[
    (df_estacion_geo_02['registros'] >= 100)
    & (df_estacion_geo_02['dias'] >= 5)
]
df_corr_geo_02 = pd.DataFrame([
    {'horizonte_h':h,'coordenada':coord, 'metrica':'sesgo_medio_c',
     'estaciones':len(t), 'pearson_por_estacion':t[coord].corr(t['sesgo_medio_c'])}
    for h in HORIZONTES_02
    for t in [elegibles_geo_02.loc[elegibles_geo_02['horizon_target_hours'].eq(h)]]
    for coord in ['latitude','longitude']
])

print('Error por hora UTC:')
display(df_horario_02.head(12).round(3))
print('\nEstados (primeros por soporte):')
display(df_estados_02.sort_values('registros',ascending=False).head(12).round(3))
print('\nCorrelación geográfica entre estaciones elegibles (no entre filas):')
display(df_corr_geo_02.round(4))

In [ ]:
# 02.9 — Figuras de hora UTC y estados, con soporte explícito
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for h in HORIZONTES_02:
    t = df_horario_02.loc[df_horario_02['horizon_target_hours'].eq(h)]
    t = t.set_index('hora_utc').reindex(range(24))
    axes[0].plot(range(24), t['mae_c'], marker='.', label=f'{h} h')
    axes[1].plot(range(24), t['sesgo_medio_c'], marker='.', label=f'{h} h')
axes[1].axhline(0,color='black',linestyle='--',linewidth=1)
for ax, titulo, ylabel in zip(axes,
    ['Magnitud del error', 'Dirección del error'],
    ['MAE (°C)','Sesgo medio (°C)']):
    ax.set_title(titulo)
    ax.set_xlabel('Hora del objetivo (UTC)')
    ax.set_ylabel(ylabel)
    ax.set_xticks(range(0,24,2))
    ax.grid(alpha=.25)
    ax.legend(title='Horizonte')
fig.suptitle('Error por hora UTC — histórico; las horas sin datos no se interpolan')
fig.tight_layout()
fig.savefig(RESULTADOS_02 / '12_error_por_hora_utc.png', dpi=170,bbox_inches='tight')
plt.show(); plt.close(fig)

# Estados con >= 1000 filas, ordenados por tamaño de sesgo (no por causalidad).
state_global = (df_bivariado.assign(
    state=df_bivariado['state'].fillna('Sin estado').replace('', 'Sin estado')
).groupby('state', as_index=False)
  .agg(registros=('error_c','size'),estaciones=('station_id','nunique'),
       dias=('fecha_objetivo','nunique'),sesgo_medio_c=('error_c','mean')))
elegibles = state_global.loc[state_global['registros']>=1000].copy()
seleccion_estado = (elegibles.assign(abs_sesgo=elegibles['sesgo_medio_c'].abs())
     .sort_values('abs_sesgo',ascending=False).head(12)
     .sort_values('sesgo_medio_c'))
fig, ax = plt.subplots(figsize=(10, 6))
if not seleccion_estado.empty:
    valores = seleccion_estado['sesgo_medio_c']
    limite = max(float(valores.abs().max()),.01)
    norm = TwoSlopeNorm(vmin=-limite,vcenter=0,vmax=limite)
    cmap = plt.get_cmap('coolwarm')
    ax.barh(seleccion_estado['state'],valores,
            color=cmap(norm(valores.to_numpy())))
ax.axvline(0,color='black',linestyle='--',linewidth=1)
ax.set(title='Estados con mayor |sesgo| entre los de soporte suficiente',
       xlabel='Sesgo medio: observado − pronosticado (°C)', ylabel='Estado')
ax.grid(axis='x',alpha=.25)
fig.tight_layout()
fig.savefig(RESULTADOS_02 / '13_sesgo_por_estado.png', dpi=170,bbox_inches='tight')
plt.show(); plt.close(fig)

## 8. Exploración visual de temperatura y síntesis académica

La nube de temperatura es una **muestra reproducible** (máximo 7.500 filas por horizonte); las estadísticas y correlaciones se calculan sobre **todas** las filas válidas. Como el target contiene algebraicamente la temperatura pronosticada, una relación observada con ella debe interpretarse con cautela. No se eligen variables del modelo final mirando el test.

Los valores numéricos se imprimen desde los datos ejecutados; no se copiaron coeficientes de la versión anterior a conclusiones Markdown. La síntesis compara asociaciones, **no importancia causal ni importancia de ML**.

In [ ]:
# 02.10 — Scatter reproducible y curvas de media por bins (sin suavizado inventado)
partes_muestra_02 = []
for h in HORIZONTES_02:
    grupo = df_bivariado.loc[df_bivariado['horizon_target_hours'].eq(h)]
    partes_muestra_02.append(grupo.sample(n=min(7500,len(grupo)),random_state=42))
df_muestra_02 = pd.concat(partes_muestra_02,ignore_index=True)

fig, ax = plt.subplots(figsize=(11, 6))
for h in HORIZONTES_02:
    t = df_muestra_02.loc[df_muestra_02['horizon_target_hours'].eq(h)]
    ax.scatter(t['temperature_forecast_c'],t['error_c'],s=5,alpha=.09,
               label=f'Puntos — {h} h')
    rangos = df_rangos_02.loc[
        df_rangos_02['variable'].eq('Temperatura pronosticada')
        & df_rangos_02['horizon_target_hours'].eq(h)
        & df_rangos_02['grupo_soporte_grafico']
    ].sort_values('orden_rango')
    # Solo graficamos centros de intervalos térmicos FINITOS (0–30 °C).
    # No inventamos centros para intervalos abiertos (<0 y ≥30).
    centros_finitos = {'0–10':5.0,'10–20':15.0,'20–30':25.0}
    r_centro = rangos.loc[rangos['rango'].isin(centros_finitos)].copy()
    if len(r_centro) >= 2:
        ax.plot(r_centro['rango'].map(centros_finitos).astype(float),
                r_centro['sesgo_medio_c'], marker='o',linewidth=2,
                label=f'Sesgo por banda finita — {h} h')
ax.axhline(0,color='black',linestyle='--',linewidth=1)
ax.set(title='Temperatura pronosticada frente a error observado (muestra)',
       xlabel='Temperatura pronosticada (°C)',
       ylabel='Error firmado: observado − pronosticado (°C)')
ax.grid(alpha=.2)
ax.legend()
fig.tight_layout()
fig.savefig(RESULTADOS_02 / '14_nube_temperatura_error.png',dpi=170,bbox_inches='tight')
plt.show();plt.close(fig)

print('Resumen de correlaciones ordenadas por |Pearson(error)|, descriptivo:')
display(df_correlaciones_02.sort_values(
    ['horizonte_h','abs_pearson_error'],ascending=[True,False]
)[['horizonte_h','variable','registros','dias','estaciones',
   'pearson_error','spearman_error','pearson_error_abs','spearman_error_abs']].round(4))

## 9. Exportación para un dashboard íntegro

El dashboard debe consumir **tablas**, no imágenes estáticas como fuentes de cálculo. Cada tabla informa horizonte, n, estaciones y días para que los filtros puedan mostrar la representatividad de un resultado. No se exportan etiquetas ni errores del test reservado. Los PNG son referencias visuales para la implementación; durante la exposición todas las vistas serán interactivas.

**Relación sugerida con las vistas:** calidad de predictores → `01`; explicación de asociaciones → `02–03`; dependencia entre predictores → `04`; hora/ubicación → `05–08`; estratos y cautela de confusión → `09`; selección visual de puntos → `10`. La interpretación final del modelo pertenece a los Notebooks 07/08 y la confianza predictiva requiere calibración independiente, no se deduce de estas correlaciones.

In [ ]:
# 02.11 — Data marts con nombres estables y manifiesto de procedencia
RESULTADOS_02.mkdir(parents=True, exist_ok=True)

df_global_02 = (df_bivariado.groupby('horizon_target_hours',as_index=False)
    .agg(registros=('error_c','size'),estaciones=('station_id','nunique'),
         dias=('fecha_objetivo','nunique'),
         sesgo_medio_c=('error_c','mean'),mae_c=('error_abs_c','mean'),
         mse_c2=('error_sq_c2','mean')))
df_global_02['rmse_c'] = np.sqrt(df_global_02.pop('mse_c2'))

exportaciones_02 = {
    '01_calidad_predictoras.csv': df_calidad_02,
    '02_correlaciones_bivariadas.csv': df_correlaciones_02,
    '03_resumen_rangos_meteorologicos.csv': df_rangos_02,
    '04_dependencias_predictores.csv': df_dependencias_02,
    '05_error_por_hora_utc.csv': df_horario_02,
    '06_error_por_estado.csv': df_estados_02,
    '07_error_por_banda_latitud.csv': df_latitud_02,
    '08_error_por_banda_longitud.csv': df_longitud_02,
    '09_humedad_por_hora_utc.csv': df_estratos_02,
    '10_muestra_temperatura_error.csv': df_muestra_02[[
        'station_id','target_time_utc','horizon_target_hours',
        'temperature_forecast_c','error_c','error_abs_c'
    ]],
    '11_correlaciones_geograficas_estaciones.csv': df_corr_geo_02,
    '12_resumen_global_horizonte.csv': df_global_02
}
for nombre,t in exportaciones_02.items():
    t.to_csv(RESULTADOS_02/nombre,index=False,encoding='utf-8-sig')

# Control cruzado: cada horizonte y cada grupo de calidad tienen misma cohorte.
assert df_global_02.set_index('horizon_target_hours')['registros'].to_dict() == {
    12:427009,24:427000
}
for nombre in VARIABLES_02:
    filtrados = df_calidad_02.loc[df_calidad_02['variable'].eq(nombre)]
    correlacion = df_correlaciones_02.loc[df_correlaciones_02['variable'].eq(nombre)]
    assert (filtrados.sort_values('horizonte_h')['registros_validos'].to_numpy()
            == correlacion.sort_values('horizonte_h')['registros'].to_numpy()).all()

resumen_02 = {
    'run_id':RUN_ID_02, 'modulo':'02_relaciones_bivariadas',
    'fuente':fuente_02, 'historico_csv':48, 'excluidos_csv':5,
    'test_reservado_csv':8, 'test_utilizado':False,
    'registros_historicos':int(len(df_bivariado)),
    'registros_12h':int(df_global_02.loc[
        df_global_02['horizon_target_hours'].eq(12),'registros'].iloc[0]),
    'registros_24h':int(df_global_02.loc[
        df_global_02['horizon_target_hours'].eq(24),'registros'].iloc[0]),
    'estaciones':int(df_bivariado['station_id'].nunique()),
    'periodo':'2026-09-10 a 2026-11-01; excluye 2026-10-22 a 2026-10-26',
    'criterio':'interval, 12/24h, temperaturas finitas; validez física por predictor',
    'errores':'error_c=observado-pronosticado; MAE=media(|error_c|)',
    'unidad_analisis':'pronóstico emparejado; filas no independientes',
    'alcance':'asociaciones descriptivas históricas; sin afirmaciones causales',
    'archivos_csv':sorted(exportaciones_02)
}
with open(RESULTADOS_02/'00_manifiesto_02.json','w',encoding='utf-8') as f:
    json.dump(resumen_02,f,ensure_ascii=False,indent=2)

for nombre in exportaciones_02:
    assert (RESULTADOS_02/nombre).is_file(),nombre

print('EXPORTACIONES:',RESULTADOS_02)
print('12 tablas CSV + JSON y gráficos PNG.')
print('Cohorte verificada:',len(df_bivariado),'| Test abierto:',resumen_02['test_utilizado'])
print('Se conservó la misma población del modelado; filtros físicos solo por análisis.')
display(df_global_02.round(4))

## 10. Conclusiones y límites antes del modelado

**Interpretaciones defendibles tras ejecutar:**

- Las tablas muestran qué variables presentan asociaciones lineales o monotónicas con el error y con su magnitud, **separadas por horizonte**. Una correlación pequeña no implica irrelevancia para un modelo no lineal.
- Los gráficos por rangos permiten localizar cambios de **sesgo** y **MAE** sin confundir ambos objetivos; deben leerse con sus recuentos de estaciones y días.
- Los predictores meteorológicos no son independientes entre sí; el análisis de dependencias anticipa límites de interpretación de Ridge y de permutation importance.
- Geografía y hora UTC son descriptores útiles, pero sus promedios reflejan mezcla de estaciones y condiciones. El Notebook 03 analiza posteriormente la hora solar.
- El presente análisis es **retrospectivo e histórico**, no demuestra causalidad física ni calibra confianza para un pronóstico futuro.

**Puente hacia el dashboard:** `resultados/dashboard/02_relaciones_bivariadas/` contiene todas las métricas por variable, horizonte, franja horaria y estado. El código de la interfaz deberá respetar los recuentos de soporte y no usar los resultados del test para rehacer este EDA.